In [ ]:
# Phase D — Exploratory Data Analysis

## Notebook 07 — Statistical Exploration

### Research Question

> **What are the statistical characteristics of the PEM fuel cell operational dataset?**

---

## Notebook Objective

The objective of this notebook is to establish a comprehensive statistical
understanding of the cleaned PEM fuel cell operational dataset.

The analysis investigates:

- dataset structure;
- variable-level statistical profiles;
- central tendency;
- dispersion and variability;
- distribution patterns;
- distribution shape;
- statistically unusual observations.

This notebook provides a statistical baseline for:

- Notebook 08 — Behaviour Exploration;
- Notebook 09 — Association Exploration;
- Notebook 10 — Engineering Relationship Exploration;
- Notebook 11 — Degradation Exploration.

No data transformation, outlier removal, engineering diagnosis, or degradation
conclusion is performed in this notebook.

In [ ]:
## Notebook Workflow

1. Configure the analytical environment.
2. Load and verify the cleaned dataset.
3. Produce a high-level dataset overview.
4. construct statistical profiles for individual variables.
5. investigate measures of central tendency.
6. quantify dispersion and relative variability.
7. examine variable distributions.
8. assess skewness and kurtosis.
9. identify statistically unusual observations.
10. consolidate statistical findings.
11. save reproducible tables and figures.
12. prepare for Notebook 08 — Behaviour Exploration.

In [2]:
# ============================================================
# 7.0.1 Import Required Libraries
# ============================================================

from pathlib import Path
import platform
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy

warnings.filterwarnings("default")

print("Required libraries imported successfully.")

Required libraries imported successfully.


In [38]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import ensure_directory

print("src package loaded successfully.")

src package loaded successfully.


In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    ensure_directory,
    validate_file,
    validate_directory,
    create_safe_filename,
    save_dataframe,
    format_file_size,
    report_saved_file
)

print("Utility functions imported successfully.")

Utility functions imported successfully.


In [40]:
# ============================================================
# Reload Project Functions
# Run this cell whenever src/eda.py is updated
# ============================================================

import importlib
import src.eda as eda

importlib.reload(eda)

from src.eda import *

print("Project functions reloaded successfully.")

Project functions reloaded successfully.


In [84]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import py_compile

py_compile.compile(
    r"src/eda.py",
    doraise=True,
)

print("eda.py syntax is valid.")

eda.py syntax is valid.


In [6]:
# ============================================================
# 7.0.2 Configure Display Settings
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 100)

np.set_printoptions(
    precision=4,
    suppress=True
)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 120,
    "axes.grid": True,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.titlesize": 13,
    "figure.autolayout": True
})

print("Display and plotting settings configured successfully.")

Display and plotting settings configured successfully.


In [7]:
# ============================================================
# 7.0.3 Define Project Paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

FIGURES_DIR = PROJECT_ROOT / "figures" / "notebook_07"
RESULTS_DIR = PROJECT_ROOT / "results" / "notebook_07"

TABLES_DIR = RESULTS_DIR / "tables"
SUMMARIES_DIR = RESULTS_DIR / "summaries"

INPUT_FILE = PROCESSED_DATA_DIR / "operational_cleaned.csv"

# Create output directories
for directory in (
    FIGURES_DIR,
    RESULTS_DIR,
    TABLES_DIR,
    SUMMARIES_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Convenience variables used throughout Notebook 07
# ------------------------------------------------------------

project_root = PROJECT_ROOT
processed_data_dir = PROCESSED_DATA_DIR

figures_dir = FIGURES_DIR
results_dir = RESULTS_DIR
tables_dir = TABLES_DIR
summaries_dir = SUMMARIES_DIR

input_file = INPUT_FILE

# Display path summary
path_summary = pd.DataFrame({
    "Purpose": [
        "Project Root",
        "Processed Data",
        "Input Dataset",
        "Figures",
        "Results",
        "Tables",
        "Summaries"
    ],
    "Path": [
        project_root,
        processed_data_dir,
        input_file,
        figures_dir,
        results_dir,
        tables_dir,
        summaries_dir
    ]
})

path_summary

,Purpose,Path
0,Project Root,C:\Users\usman\Desktop\PEMFC_Dissertation
1,Processed Data,C:\Users\usman\Desktop\PEMFC_Dissertation\data\processed
2,Input Dataset,C:\Users\usman\Desktop\PEMFC_Dissertation\data\processed\operational_cleaned.csv
3,Figures,C:\Users\usman\Desktop\PEMFC_Dissertation\figures\notebook_07
4,Results,C:\Users\usman\Desktop\PEMFC_Dissertation\results\notebook_07
5,Tables,C:\Users\usman\Desktop\PEMFC_Dissertation\results\notebook_07\tables
6,Summaries,C:\Users\usman\Desktop\PEMFC_Dissertation\results\notebook_07\summaries


In [4]:
# ============================================================
# 7.0.4 Validate Required Paths
# ============================================================

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root was not found: {PROJECT_ROOT}"
    )

if not PROCESSED_DATA_DIR.exists():
    raise FileNotFoundError(
        f"Processed-data directory was not found: "
        f"{PROCESSED_DATA_DIR}"
    )

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Cleaned operational dataset was not found: "
        f"{INPUT_FILE}"
    )

print("All required project paths were verified successfully.")

All required project paths were verified successfully.


In [8]:
# ============================================================
# 7.0.5 Record Computing Environment
# ============================================================

environment_summary = pd.DataFrame({
    "Component": [
        "Operating system",
        "Python version",
        "NumPy version",
        "pandas version",
        "SciPy version",
        "Matplotlib version"
    ],
    "Version": [
        platform.platform(),
        sys.version.split()[0],
        np.__version__,
        pd.__version__,
        scipy.__version__,
        plt.matplotlib.__version__
    ]
})

environment_summary

,Component,Version
0,Operating system,Windows-11-10.0.26200-SP0
1,Python version,3.13.11
2,NumPy version,2.4.6
3,pandas version,3.0.3
4,SciPy version,1.18.0
5,Matplotlib version,3.11.0


In [9]:
environment_summary.to_csv(
    TABLES_DIR / "07_environment_summary.csv",
    index=False
)

print("Environment summary saved successfully.")

Environment summary saved successfully.


In [ ]:
## 7.0.6 Load the Cleaned Operational Dataset

The cleaned operational dataset produced during the data-preparation phase is
loaded without modifying the source file.

The dataset is treated as read-only throughout Notebook 07. Statistical
exploration may generate tables, figures, and summaries, but it must not alter
the original cleaned measurements.

In [10]:
# ============================================================
# 7.0.6 Load the Cleaned Operational Dataset
# ============================================================

cleaned_dataset = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

print("Cleaned operational dataset loaded successfully.")
print(f"Observations: {cleaned_dataset.shape[0]:,}")
print(f"Variables: {cleaned_dataset.shape[1]:,}")

Cleaned operational dataset loaded successfully.
Observations: 3,629,680
Variables: 18


In [11]:
# ============================================================
# 7.0.7 Create Working Copy
# ============================================================

df = cleaned_dataset.copy(deep=True)

print("A working copy of the cleaned dataset was created.")

A working copy of the cleaned dataset was created.


In [12]:
# ============================================================
# 7.0.8 Verify Successful Dataset Loading
# ============================================================

required_columns = {
    "time",
    "current",
    "voltage",
    "power",
    "pressure_anode_inlet",
    "pressure_anode_outlet",
    "pressure_cathode_inlet",
    "pressure_cathode_outlet",
    "temp_anode_endplate",
    "temp_anode_dewpoint_water",
    "temp_anode_inlet",
    "temp_anode_outlet",
    "temp_cathode_dewpoint_water",
    "temp_cathode_inlet",
    "temp_cathode_outlet",
    "total_anode_stack_flow",
    "total_cathode_stack_flow",
    "operating_hour"
}

missing_required_columns = sorted(
    required_columns.difference(df.columns)
)

if missing_required_columns:
    raise ValueError(
        "The dataset is missing required columns:\n"
        + "\n".join(missing_required_columns)
    )

if df.empty:
    raise ValueError("The loaded dataset contains no observations.")

if df.columns.duplicated().any():
    duplicated_columns = df.columns[
        df.columns.duplicated()
    ].tolist()

    raise ValueError(
        f"Duplicated column names were detected: "
        f"{duplicated_columns}"
    )

print("Dataset structure verified successfully.")

Dataset structure verified successfully.


In [18]:
# ============================================================
# 7.0.9 Dataset Loading Summary
# ============================================================

loading_summary = pd.DataFrame({
    "Metric": [
        "Input file",
        "Total observations",
        "Total variables",
        "Numeric variables",
        "Non-numeric variables",
        "Missing observations",
        "Duplicate rows",
        "Operating-hour experiments"
    ],
    "Value": [
        INPUT_FILE.name,
        f"{df.shape[0]:,}",
        df.shape[1],
        df.select_dtypes(include=np.number).shape[1],
        df.select_dtypes(exclude=np.number).shape[1],
        f"{df.isna().sum().sum():,}",
        f"{df.duplicated().sum():,}",
        (
            df["operating_hour"].nunique()
            if "operating_hour" in df.columns
            else "Not available"
        )
    ]
})

loading_summary

,Metric,Value
0,Input file,operational_cleaned.csv
1,Total observations,"3,629,680"
2,Total variables,18
3,Numeric variables,18
4,Non-numeric variables,0
5,Missing observations,0
6,Duplicate rows,0
7,Operating-hour experiments,20


In [19]:
loading_summary.to_csv(
    TABLES_DIR / "07_dataset_loading_summary.csv",
    index=False
)

print("Dataset loading summary saved successfully.")

Dataset loading summary saved successfully.


In [16]:
# ============================================================
# 7.0.10 Reproducibility Checkpoint
# ============================================================

assert df.shape == cleaned_dataset.shape
assert df.columns.equals(cleaned_dataset.columns)
assert len(df) > 0
assert df.columns.is_unique

print("Notebook setup completed successfully.")
print("The dataset is ready for Section 7.1 — Dataset Overview.")

Notebook setup completed successfully.
The dataset is ready for Section 7.1 — Dataset Overview.


In [ ]:
# 7.1 Dataset Overview

## Purpose

Before performing any statistical analysis, it is essential to obtain a
high-level overview of the cleaned operational dataset.

This section verifies the overall characteristics of the dataset and confirms
that the correct dataset has been loaded for statistical exploration.

The investigation focuses on:

- dataset dimensions;
- variable names;
- data types;
- memory usage;
- missing values;
- duplicate observations.

No statistical interpretation or engineering conclusions are drawn at this
stage. The objective is simply to establish confidence that the dataset is
complete, correctly structured, and suitable for subsequent statistical
analysis.

In [ ]:
## 7.1.1 Dataset Dimensions

Dataset dimensions provide an overview of the amount of information available
for analysis.

Understanding the number of observations and variables allows researchers to
assess the scale of the dataset and provides context for subsequent statistical
investigations.

In [20]:
# ============================================================
# 7.1.1 Dataset Dimensions
# ============================================================

dataset_dimensions = pd.DataFrame({
    "Metric": [
        "Number of observations",
        "Number of variables"
    ],
    "Value": [
        f"{df.shape[0]:,}",
        df.shape[1]
    ]
})

dataset_dimensions

,Metric,Value
0,Number of observations,"3,629,680"
1,Number of variables,18


In [21]:
dataset_dimensions.to_csv(
    TABLES_DIR / "07_dataset_dimensions.csv",
    index=False
)

In [ ]:
## 7.1.2 Variable List

Before analysing the operational measurements, every recorded variable should
be identified.

This provides a complete inventory of the available measurements and ensures
that all variables are included in the statistical exploration.

In [22]:
# ============================================================
# 7.1.2 Variable List
# ============================================================

variable_list = pd.DataFrame({
    "Variable": df.columns,
    "Data Type": df.dtypes.values
})

variable_list

,Variable,Data Type
0,operating_hour,int64
1,time,float64
2,current,float64
3,voltage,float64
4,power,float64
5,pressure_anode_inlet,float64
6,pressure_anode_outlet,float64
7,pressure_cathode_inlet,float64
8,pressure_cathode_outlet,float64
9,temp_anode_endplate,float64


In [23]:
variable_list.to_csv(
    TABLES_DIR / "07_variable_list.csv",
    index=False
)

In [ ]:
## 7.1.3 Data Type Summary

The statistical methods used throughout this notebook require numerical
variables.

This section confirms the distribution of data types within the cleaned
dataset.

In [24]:
# ============================================================
# 7.1.3 Data Type Summary
# ============================================================

dtype_summary = (
    df.dtypes
      .value_counts()
      .rename_axis("Data Type")
      .reset_index(name="Count")
)

dtype_summary

,Data Type,Count
0,float64,17
1,int64,1


In [25]:
dtype_summary.to_csv(
    TABLES_DIR / "07_dtype_summary.csv",
    index=False
)

In [ ]:
## 7.1.4 Memory Usage

Understanding the memory requirements of the operational dataset is useful for
planning subsequent analyses and assessing computational requirements.

In [26]:
# ============================================================
# 7.1.4 Memory Usage
# ============================================================

memory_usage = pd.DataFrame({
    "Variable": df.columns,
    "Memory (MB)": (
        df.memory_usage(deep=True)[1:] / (1024 ** 2)
    ).round(2)
})

memory_usage

,Variable,Memory (MB)
operating_hour,operating_hour,27.69
time,time,27.69
current,current,27.69
voltage,voltage,27.69
power,power,27.69
pressure_anode_inlet,pressure_anode_inlet,27.69
pressure_anode_outlet,pressure_anode_outlet,27.69
pressure_cathode_inlet,pressure_cathode_inlet,27.69
pressure_cathode_outlet,pressure_cathode_outlet,27.69
temp_anode_endplate,temp_anode_endplate,27.69


In [27]:
total_memory_mb = (
    df.memory_usage(deep=True).sum() / (1024 ** 2)
)

print(f"Total Dataset Memory: {total_memory_mb:.2f} MB")

Total Dataset Memory: 498.46 MB


In [28]:
memory_usage.to_csv(
    TABLES_DIR / "07_memory_usage.csv",
    index=False
)

In [29]:
# ============================================================
# 7.1.5 Missing Value Summary
# ============================================================

missing_summary = pd.DataFrame({
    "Variable": df.columns,
    "Missing Values": df.isna().sum().values,
    "Missing (%)": (
        df.isna().mean() * 100
    ).round(4).values
})

missing_summary

,Variable,Missing Values,Missing (%)
0,operating_hour,0,0.0
1,time,0,0.0
2,current,0,0.0
3,voltage,0,0.0
4,power,0,0.0
5,pressure_anode_inlet,0,0.0
6,pressure_anode_outlet,0,0.0
7,pressure_cathode_inlet,0,0.0
8,pressure_cathode_outlet,0,0.0
9,temp_anode_endplate,0,0.0


In [30]:
missing_summary.to_csv(
    TABLES_DIR / "07_missing_summary.csv",
    index=False
)

In [31]:
# ============================================================
# 7.1.6 Duplicate Summary
# ============================================================

duplicate_summary = pd.DataFrame({
    "Metric": [
        "Duplicate observations"
    ],
    "Value": [
        df.duplicated().sum()
    ]
})

duplicate_summary

,Metric,Value
0,Duplicate observations,0


In [32]:
duplicate_summary.to_csv(
    TABLES_DIR / "07_duplicate_summary.csv",
    index=False
)

In [33]:
# ============================================================
# 7.1.7 Dataset Overview Summary
# ============================================================

# Calculate dataset-level metrics
total_memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
numeric_variables = df.select_dtypes(include=np.number).shape[1]
non_numeric_variables = df.select_dtypes(exclude=np.number).shape[1]

overview_summary = pd.DataFrame({
    "Metric": [
        "Dataset loaded successfully",
        "Number of observations",
        "Number of variables",
        "Numeric variables",
        "Non-numeric variables",
        "Operating-hour experiments",
        "Total dataset memory",
        "Missing observations",
        "Duplicate observations"
    ],
    "Value": [
        "Yes",
        f"{df.shape[0]:,}",
        df.shape[1],
        numeric_variables,
        non_numeric_variables,
        df["operating_hour"].nunique() if "operating_hour" in df.columns else "N/A",
        f"{total_memory_mb:.2f} MB",
        f"{df.isna().sum().sum():,}",
        f"{df.duplicated().sum():,}"
    ]
})

display(overview_summary)

,Metric,Value
0,Dataset loaded successfully,Yes
1,Number of observations,"3,629,680"
2,Number of variables,18
3,Numeric variables,18
4,Non-numeric variables,0
5,Operating-hour experiments,20
6,Total dataset memory,498.46 MB
7,Missing observations,0
8,Duplicate observations,0


In [34]:
overview_summary.to_csv(
    SUMMARIES_DIR / "07_dataset_overview_summary.csv",
    index=False
)

print("Dataset overview summary saved successfully.")

Dataset overview summary saved successfully.


In [ ]:
### Research Insight 7.1

The dataset overview confirms that the expected cleaned operational dataset has
been successfully loaded and retains its validated structure. The variables,
data types, and dataset dimensions are consistent with the outputs of the data
preparation phase, while the absence of missing values and duplicate
observations provides confidence that the statistical analyses performed in the
following sections will be based on a complete and reliable dataset.

This verification establishes the foundation for the detailed statistical
characterisation undertaken in the subsequent sections of Notebook 07.

In [ ]:
# 7.2 Variable Statistical Profiles

## Purpose

Before comparing variables statistically, each operational measurement should
be examined individually.

A Variable Statistical Profile provides a concise summary describing the
statistical characteristics of a single variable.

Each profile includes:

- engineering description;
- data type;
- number of observations;
- missing values;
- descriptive statistics;
- histogram;
- boxplot;
- initial statistical observations.

These profiles provide a reference for all subsequent exploratory analyses and
can be revisited throughout the dissertation.

In [35]:
# ============================================================
# 7.2.1 Variable Metadata
# ============================================================

variable_metadata = {

    "time": {
        "Description": "Elapsed experimental time",
        "Unit": "seconds"
    },

    "current": {
        "Description": "Stack current",
        "Unit": "A"
    },

    "voltage": {
        "Description": "Stack voltage",
        "Unit": "V"
    },

    "power": {
        "Description": "Electrical power",
        "Unit": "W"
    },

    "pressure_anode_inlet": {
        "Description": "Anode inlet pressure",
        "Unit": "kPaG"
    },

    "pressure_anode_outlet": {
        "Description": "Anode outlet pressure",
        "Unit": "kPaG"
    },

    "pressure_cathode_inlet": {
        "Description": "Cathode inlet pressure",
        "Unit": "kPaG"
    },

    "pressure_cathode_outlet": {
        "Description": "Cathode outlet pressure",
        "Unit": "kPaG"
    },

    "temp_anode_endplate": {
        "Description": "Anode endplate temperature",
        "Unit": "°C"
    },

    "temp_anode_dewpoint_water": {
        "Description": "Anode dewpoint temperature",
        "Unit": "°C"
    },

    "temp_anode_inlet": {
        "Description": "Anode inlet temperature",
        "Unit": "°C"
    },

    "temp_anode_outlet": {
        "Description": "Anode outlet temperature",
        "Unit": "°C"
    },

    "temp_cathode_dewpoint_water": {
        "Description": "Cathode dewpoint temperature",
        "Unit": "°C"
    },

    "temp_cathode_inlet": {
        "Description": "Cathode inlet temperature",
        "Unit": "°C"
    },

    "temp_cathode_outlet": {
        "Description": "Cathode outlet temperature",
        "Unit": "°C"
    },

    "total_anode_stack_flow": {
        "Description": "Total anode stack flow",
        "Unit": "SLPM"
    },

    "total_cathode_stack_flow": {
        "Description": "Total cathode stack flow",
        "Unit": "SLPM"
    }

}

In [ ]:
# C12.1 Variable Profile Analysis

## Objective

The purpose of this section is to perform a comprehensive univariate analysis of each operational variable in the PEMFC dataset.

Each variable profile consists of:

- Variable metadata
- Descriptive statistical summary
- Histogram
- Boxplot

These analyses provide an understanding of the distribution, variability, central tendency, dispersion, skewness, kurtosis and potential outliers present in each operational parameter.

The findings from this section establish the statistical characteristics of the dataset before proceeding to relationship analysis and feature engineering.

In [ ]:
## Interpretation

The metadata confirms that the variable represents stack current measured in amperes.

The statistical summary provides information regarding:

- number of observations
- missing values
- variability
- central tendency
- spread
- distribution shape
- potential outliers

These statistics provide the quantitative basis for understanding how current behaves throughout the ageing experiment.

In [ ]:
## Variable Under Investigation

### Current

Current represents the electrical current drawn from the PEM fuel cell stack during operation.

It is one of the principal operating variables because it directly determines the electrochemical load imposed on the stack and strongly influences voltage behaviour, power output, reactant consumption, water management and degradation mechanisms.

Unit: Ampere (A)

In [41]:
# ----------------------------------------------------------
# Generate the variable profile for stack current
# ----------------------------------------------------------

current_profile = variable_profile(
    dataframe=df,
    variable="current",
    description="Stack current",
    unit="A",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

In [169]:
current_outliers = cleaned_dataset[
    cleaned_dataset["current"] > 34.4001
]

print(len(current_outliers))

print(current_outliers["current"].value_counts().head(20))

135333
current
35.5318    50186
35.5342    41200
35.5366    20080
35.5294    13360
35.5390     9693
35.5414      692
35.5270      110
35.5246        8
35.5174        2
35.5222        2
Name: count, dtype: int64


In [94]:
# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(current_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(current_profile["statistics"])

,Property,Value
0,Variable,current
1,Description,Stack current
2,Unit,A
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,113
5,Minimum,-0.0025 A
6,Q1,1.7604 A
7,Mean,9.9364 A
8,Median,9.4848 A
9,Mode,1.7604 A


In [ ]:
## C12.1.1 Current (A)

### Statistical Exploration and Engineering Interpretation

Current is one of the most important operational variables in a proton exchange membrane fuel cell (PEMFC), as it represents the electrical load demanded from the fuel cell stack. Variations in current directly influence electrochemical reaction rates, hydrogen and oxygen consumption, heat generation, water production, and ultimately the degradation behaviour of the fuel cell. Consequently, understanding the statistical characteristics of current is essential before investigating relationships with other operational parameters and developing predictive machine learning models.

In [ ]:
### Findings

The descriptive statistics indicate that the current variable consists of **3,629,680 valid observations**, with **no missing values**, demonstrating excellent data completeness and eliminating the need for missing value treatment. The dataset contains **113 unique current levels**, suggesting that the fuel cell was operated under predefined loading conditions rather than continuously varying electrical demand.

The current ranges from **−0.0025 A** to **35.5414 A**, producing a total operating range of **35.5439 A**. The minimum value is extremely close to zero and is likely associated with measurement noise or transient start-up/shutdown behaviour rather than reverse current operation.

The **mean current (9.9364 A)** is slightly higher than the **median (9.4848 A)**, while the **mode is 1.7604 A**. This indicates that the lowest operating current occurred most frequently throughout the ageing experiment. The noticeable difference between the mode and both the mean and median reflects the multi-level operating strategy employed during testing.

The **standard deviation (9.3921 A)** and **coefficient of variation (94.52%)** indicate considerable variability in current. Such high variability is expected because the experimental protocol intentionally subjected the fuel cell to multiple loading conditions to evaluate performance degradation across different operating regimes.

The **interquartile range (13.0559 A)** further confirms the wide spread of the operational current values. The first quartile (1.7604 A) and third quartile (14.8163 A) show that fifty percent of the observations lie within a broad current interval, reflecting the extensive operational envelope investigated during the durability experiment.

The current distribution exhibits a **positive skewness (1.0395)**, indicating that observations are concentrated at lower current values with a longer tail extending toward higher loads. The **kurtosis (0.3472)** is close to zero, suggesting that the distribution is only slightly more peaked than a normal distribution and does not exhibit excessive heavy tails.

The IQR analysis identified **135,333 observations (3.73%)** as statistical outliers, all exceeding the calculated upper bound of **34.4001 A**. Inspection of the histogram and boxplot indicates that these observations correspond primarily to the highest operating current level (approximately 35.5 A). Since these measurements represent genuine operating conditions deliberately applied during testing, they should be retained rather than removed during data preprocessing.

In [ ]:
### Engineering Interpretation

The histogram clearly demonstrates that current is **multimodal**, with several distinct peaks corresponding to discrete operating current levels rather than a continuous distribution. This observation strongly suggests that the ageing experiment followed a predefined loading protocol in which the PEMFC stack was repeatedly operated at specific current setpoints.

Lower current levels appear considerably more frequently than higher loads, indicating that a substantial proportion of the experiment was conducted under relatively mild operating conditions, while higher currents were applied periodically to evaluate stack behaviour under increased electrochemical stress.

The presence of multiple operating regimes is advantageous for machine learning because it exposes the model to a wide range of operational conditions, thereby improving its ability to learn the relationship between operating current and performance degradation. Furthermore, the absence of missing data and the limited proportion of outliers indicate that current is a high-quality predictor suitable for subsequent correlation analysis, feature engineering, and degradation modelling.

In [ ]:
### Key Findings

- No missing observations were identified.
- The dataset contains 113 discrete operating current levels.
- Current follows a multimodal rather than a normal distribution.
- High variability (CV = 94.52%) reflects the experimental loading protocol.
- Approximately 3.73% of observations are classified as statistical outliers, representing legitimate high-current operating conditions rather than erroneous measurements.
- Current is expected to be one of the most influential predictors of PEMFC voltage behaviour, power generation, reactant consumption, and degradation.

In [ ]:
## Variable Under Investigation

### Voltage

### Variable Overview

In [47]:
# ----------------------------------------------------------
# Generate the variable profile for stack voltage
# ----------------------------------------------------------

voltage_profile = variable_profile(
    dataframe=df,
    variable="voltage",
    description="Stack voltage",
    unit="V",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(voltage_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(voltage_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,voltage
1,Description,Stack voltage
2,Unit,V
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,"1,437"
5,Minimum,-0.0587 V
6,Q1,0.6871 V
7,Mean,0.7567 V
8,Median,0.7511 V
9,Mode,0.8409 V


In [49]:
# ----------------------------------------------------------
# Generate the variable profile for stack power
# ----------------------------------------------------------

power_profile = variable_profile(
    dataframe=df,
    variable="power",
    description="Stack power",
    unit="W",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(power_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(power_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,power
1,Description,Stack power
2,Unit,W
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,"1,488"
5,Minimum,-0.8700 W
6,Q1,1.4900 W
7,Mean,6.6330 W
8,Median,6.9300 W
9,Mode,1.4800 W


In [42]:
# ----------------------------------------------------------
# Generate the variable profile for anode inlet pressure
# ----------------------------------------------------------

pressure_anode_inlet_profile = variable_profile(
    dataframe=df,
    variable="pressure_anode_inlet",
    description="Anode inlet pressure",
    unit="kPaG",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(pressure_anode_inlet_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(pressure_anode_inlet_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,pressure_anode_inlet
1,Description,Anode inlet pressure
2,Unit,kPaG
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,519
5,Minimum,49.0699 kPaG
6,Q1,109.9013 kPaG
7,Mean,109.9550 kPaG
8,Median,109.9013 kPaG
9,Mode,109.9013 kPaG


In [43]:
# ----------------------------------------------------------
# Generate the variable profile for anode outlet pressure
# ----------------------------------------------------------

pressure_anode_outlet_profile = variable_profile(
    dataframe=df,
    variable="pressure_anode_outlet",
    description="Anode outlet pressure",
    unit="kPaG",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(pressure_anode_outlet_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(pressure_anode_outlet_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,pressure_anode_outlet
1,Description,Anode outlet pressure
2,Unit,kPaG
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,510
5,Minimum,49.4387 kPaG
6,Q1,110.0239 kPaG
7,Mean,110.1856 kPaG
8,Median,110.3273 kPaG
9,Mode,110.4284 kPaG


In [44]:
# ----------------------------------------------------------
# Generate the variable profile for cathode inlet pressure
# ----------------------------------------------------------

pressure_cathode_inlet_profile = variable_profile(
    dataframe=df,
    variable="pressure_cathode_inlet",
    description="Cathode inlet pressure",
    unit="kPaG",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(pressure_cathode_inlet_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(pressure_cathode_inlet_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,pressure_cathode_inlet
1,Description,Cathode inlet pressure
2,Unit,kPaG
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,167
5,Minimum,100.4794 kPaG
6,Q1,109.3954 kPaG
7,Mean,109.8189 kPaG
8,Median,109.8001 kPaG
9,Mode,109.5978 kPaG


In [45]:
# ----------------------------------------------------------
# Generate the variable profile for cathode outlet pressure
# ----------------------------------------------------------

pressure_cathode_outlet_profile = variable_profile(
    dataframe=df,
    variable="pressure_cathode_outlet",
    description="Cathode outlet pressure",
    unit="kPaG",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(pressure_cathode_outlet_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(pressure_cathode_outlet_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,pressure_cathode_outlet
1,Description,Cathode outlet pressure
2,Unit,kPaG
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,85
5,Minimum,99.2598 kPaG
6,Q1,107.0705 kPaG
7,Mean,107.7650 kPaG
8,Median,108.3870 kPaG
9,Mode,108.7921 kPaG


In [50]:
# ----------------------------------------------------------
# Generate the variable profile for anode endplate temperature
# ----------------------------------------------------------

temp_anode_endplate_profile = variable_profile(
    dataframe=df,
    variable="temp_anode_endplate",
    description="Anode endplate temperature",
    unit="°C",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(temp_anode_endplate_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(temp_anode_endplate_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,temp_anode_endplate
1,Description,Anode endplate temperature
2,Unit,°C
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,"108,594"
5,Minimum,81.4062 °C
6,Q1,82.9359 °C
7,Mean,83.4078 °C
8,Median,83.5750 °C
9,Mode,83.6715 °C


In [51]:
# ----------------------------------------------------------
# Generate the variable profile for anode dew point temperature
# ----------------------------------------------------------

temp_anode_dewpoint_water_profile = variable_profile(
    dataframe=df,
    variable="temp_anode_dewpoint_water",
    description="Anode dew point water temperature",
    unit="°C",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(temp_anode_dewpoint_water_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(temp_anode_dewpoint_water_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,temp_anode_dewpoint_water
1,Description,Anode dew point water temperature
2,Unit,°C
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,"64,261"
5,Minimum,52.3397 °C
6,Q1,54.9457 °C
7,Mean,54.9523 °C
8,Median,54.9929 °C
9,Mode,54.9998 °C


In [52]:
# ----------------------------------------------------------
# Generate the variable profile for anode inlet temperature
# ----------------------------------------------------------

temp_anode_inlet_profile = variable_profile(
    dataframe=df,
    variable="temp_anode_inlet",
    description="Anode inlet temperature",
    unit="°C",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(temp_anode_inlet_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(temp_anode_inlet_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,temp_anode_inlet
1,Description,Anode inlet temperature
2,Unit,°C
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,"115,415"
5,Minimum,63.1064 °C
6,Q1,69.7354 °C
7,Mean,69.9828 °C
8,Median,69.9372 °C
9,Mode,69.8949 °C


In [58]:
# ----------------------------------------------------------
# Generate the variable profile for anode outlet temperature
# ----------------------------------------------------------

temp_anode_outlet_profile = variable_profile(
    dataframe=df,
    variable="temp_anode_outlet",
    description="Anode outlet temperature",
    unit="°C",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(temp_anode_outlet_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(temp_anode_outlet_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,temp_anode_outlet
1,Description,Anode outlet temperature
2,Unit,°C
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,"509,833"
5,Minimum,25.9413 °C
6,Q1,37.2700 °C
7,Mean,41.0738 °C
8,Median,41.0224 °C
9,Mode,40.9630 °C


In [53]:
# ----------------------------------------------------------
# Generate the variable profile for cathode dew point temperature
# ----------------------------------------------------------

temp_cathode_dewpoint_water_profile = variable_profile(
    dataframe=df,
    variable="temp_cathode_dewpoint_water",
    description="Cathode dew point water temperature",
    unit="°C",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(temp_cathode_dewpoint_water_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(temp_cathode_dewpoint_water_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,temp_cathode_dewpoint_water
1,Description,Cathode dew point water temperature
2,Unit,°C
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,"44,269"
5,Minimum,58.7906 °C
6,Q1,64.8940 °C
7,Mean,64.9552 °C
8,Median,64.9875 °C
9,Mode,65.0105 °C


In [54]:
# ----------------------------------------------------------
# Generate the variable profile for cathode inlet temperature
# ----------------------------------------------------------

temp_cathode_inlet_profile = variable_profile(
    dataframe=df,
    variable="temp_cathode_inlet",
    description="Cathode inlet temperature",
    unit="°C",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(temp_cathode_inlet_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(temp_cathode_inlet_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,temp_cathode_inlet
1,Description,Cathode inlet temperature
2,Unit,°C
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,"140,323"
5,Minimum,67.1977 °C
6,Q1,69.3884 °C
7,Mean,69.9927 °C
8,Median,69.7530 °C
9,Mode,69.7017 °C


In [60]:
# ----------------------------------------------------------
# Generate the variable profile for cathode outlet temperature
# ----------------------------------------------------------

temp_cathode_outlet_profile = variable_profile(
    dataframe=df,
    variable="temp_cathode_outlet",
    description="Cathode outlet temperature",
    unit="°C",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(temp_cathode_outlet_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(temp_cathode_outlet_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,temp_cathode_outlet
1,Description,Cathode outlet temperature
2,Unit,°C
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,"463,978"
5,Minimum,46.1108 °C
6,Q1,53.9469 °C
7,Mean,55.9852 °C
8,Median,55.8761 °C
9,Mode,54.6281 °C


In [55]:
# ----------------------------------------------------------
# Generate the variable profile for total anode stack flow
# ----------------------------------------------------------

total_anode_stack_flow_profile = variable_profile(
    dataframe=df,
    variable="total_anode_stack_flow",
    description="Total anode stack flow",
    unit="L/min",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(total_anode_stack_flow_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(total_anode_stack_flow_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,total_anode_stack_flow
1,Description,Total anode stack flow
2,Unit,L/min
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,353
5,Minimum,0.0690 L/min
6,Q1,0.0840 L/min
7,Mean,0.1656 L/min
8,Median,0.1330 L/min
9,Mode,0.0840 L/min


In [56]:
# ----------------------------------------------------------
# Generate the variable profile for total cathode stack flow
# ----------------------------------------------------------

total_cathode_stack_flow_profile = variable_profile(
    dataframe=df,
    variable="total_cathode_stack_flow",
    description="Total cathode stack flow",
    unit="L/min",
    bins=50,
    output_directory=TABLES_DIR,
    save_outputs=False,
    show_plots=True,
)

# ----------------------------------------------------------
# Display the metadata table
# ----------------------------------------------------------

display(total_cathode_stack_flow_profile["metadata"])

# ----------------------------------------------------------
# Display the statistical summary
# ----------------------------------------------------------

display(total_cathode_stack_flow_profile["statistics"])

<Figure size 1200x720 with 1 Axes>

<Figure size 1200x480 with 1 Axes>

,Property,Value
0,Variable,total_cathode_stack_flow
1,Description,Total cathode stack flow
2,Unit,L/min
3,Data Type,float64


,Metric,Value
0,Total Observations,"3,629,680"
1,Valid Observations,"3,629,680"
2,Missing Values,0
3,Missing (%),0.00%
4,Unique Values,"1,124"
5,Minimum,0.2900 L/min
6,Q1,0.3490 L/min
7,Mean,0.6902 L/min
8,Median,0.5530 L/min
9,Mode,0.3490 L/min


In [61]:
### 7.4.1 Statistical Summary Table
# ============================================================
# Master Statistical Summary
# ============================================================

profiles = {
    "Current": current_profile,
    "Voltage": voltage_profile,
    "Power": power_profile,
    "Pressure Anode Inlet": pressure_anode_inlet_profile,
    "Pressure Anode Outlet": pressure_anode_outlet_profile,
    "Pressure Cathode Inlet": pressure_cathode_inlet_profile,
    "Pressure Cathode Outlet": pressure_cathode_outlet_profile,
    "Temperature Anode Endplate": temp_anode_endplate_profile,
    "Temperature Anode Dewpoint": temp_anode_dewpoint_water_profile,
    "Temperature Anode Inlet": temp_anode_inlet_profile,
    "Temperature Anode Outlet": temp_anode_outlet_profile,
    "Temperature Cathode Dewpoint": temp_cathode_dewpoint_water_profile,
    "Temperature Cathode Inlet": temp_cathode_inlet_profile,
    "Temperature Cathode Outlet": temp_cathode_outlet_profile,
    "Total Anode Stack Flow": total_anode_stack_flow_profile,
    "Total Cathode Stack Flow": total_cathode_stack_flow_profile,
}

statistics_summary = []

for name, profile in profiles.items():

    stats = profile["statistics"].copy()

    stats.columns = ["Metric", "Value"]

    row = {"Variable": name}

    for _, r in stats.iterrows():
        row[r["Metric"]] = r["Value"]

    statistics_summary.append(row)

statistics_summary = pd.DataFrame(statistics_summary)

display(statistics_summary)
statistics_summary=statistics_summary.copy()

,Variable,Total Observations,Valid Observations,Missing Values,Missing (%),Unique Values,Minimum,Q1,Mean,Median,Mode,Q3,Maximum,Range,Variance,Standard Deviation,IQR,Coefficient of Variation (%),Skewness,Kurtosis,IQR Lower Bound,IQR Upper Bound,IQR Outlier Count,IQR Outliers (%)
0,Current,"3,629,680","3,629,680",0,0.00%,113,-0.0025 A,1.7604 A,9.9364 A,9.4848 A,1.7604 A,14.8163 A,35.5414 A,35.5439 A,88.210841 A²,9.3921 A,13.0559 A,94.52%,1.0395,0.3472,-17.8234 A,34.4001 A,"135,333",3.728511%
1,Voltage,"3,629,680","3,629,680",0,0.00%,"1,437",-0.0587 V,0.6871 V,0.7567 V,0.7511 V,0.8409 V,0.8428 V,0.9498 V,1.0085 V,0.009453 V²,0.0972 V,0.1557 V,12.85%,-0.4867,-0.6132,0.4536 V,1.0763 V,4,0.000110%
2,Power,"3,629,680","3,629,680",0,0.00%,"1,488",-0.8700 W,1.4900 W,6.6330 W,6.9300 W,1.4800 W,10.3300 W,21.7800 W,22.6500 W,29.867930 W²,5.4652 W,8.8400 W,82.39%,0.6275,-0.7118,-11.7700 W,23.5900 W,0,0.000000%
3,Pressure Anode Inlet,"3,629,680","3,629,680",0,0.00%,519,49.0699 kPaG,109.9013 kPaG,109.9550 kPaG,109.9013 kPaG,109.9013 kPaG,110.2048 kPaG,126.7975 kPaG,77.7277 kPaG,1.533169 kPaG²,1.2382 kPaG,0.3035 kPaG,1.13%,-23.9786,830.5370,109.4460 kPaG,110.6601 kPaG,"280,556",7.729497%
4,Pressure Anode Outlet,"3,629,680","3,629,680",0,0.00%,510,49.4387 kPaG,110.0239 kPaG,110.1856 kPaG,110.3273 kPaG,110.4284 kPaG,110.4284 kPaG,126.2002 kPaG,76.7615 kPaG,1.598050 kPaG²,1.2641 kPaG,0.4044 kPaG,1.15%,-21.9322,745.6754,109.4173 kPaG,111.0350 kPaG,"302,492",8.333848%
5,Pressure Cathode Inlet,"3,629,680","3,629,680",0,0.00%,167,100.4794 kPaG,109.3954 kPaG,109.8189 kPaG,109.8001 kPaG,109.5978 kPaG,110.5084 kPaG,127.8093 kPaG,27.3299 kPaG,2.221347 kPaG²,1.4904 kPaG,1.1129 kPaG,1.36%,-1.8667,6.3340,107.7260 kPaG,112.1777 kPaG,"315,950",8.704624%
6,Pressure Cathode Outlet,"3,629,680","3,629,680",0,0.00%,85,99.2598 kPaG,107.0705 kPaG,107.7650 kPaG,108.3870 kPaG,108.7921 kPaG,108.7921 kPaG,111.3239 kPaG,12.0641 kPaG,2.550717 kPaG²,1.5971 kPaG,1.7216 kPaG,1.48%,-1.7283,2.6537,104.4880 kPaG,111.3746 kPaG,"217,194",5.983833%
7,Temperature Anode Endplate,"3,629,680","3,629,680",0,0.00%,"108,594",81.4062 °C,82.9359 °C,83.4078 °C,83.5750 °C,83.6715 °C,83.7908 °C,84.9783 °C,3.5722 °C,0.306581 °C²,0.5537 °C,0.8549 °C,0.66%,-0.4316,-0.5413,81.6535 °C,85.0732 °C,128,0.003526%
8,Temperature Anode Dewpoint,"3,629,680","3,629,680",0,0.00%,"64,261",52.3397 °C,54.9457 °C,54.9523 °C,54.9929 °C,54.9998 °C,55.0298 °C,56.5250 °C,4.1853 °C,0.043535 °C²,0.2086 °C,0.0841 °C,0.38%,-2.8794,14.0682,54.8196 °C,55.1560 °C,"588,599",16.216278%
9,Temperature Anode Inlet,"3,629,680","3,629,680",0,0.00%,"115,415",63.1064 °C,69.7354 °C,69.9828 °C,69.9372 °C,69.8949 °C,70.1880 °C,76.7344 °C,13.6280 °C,0.190779 °C²,0.4368 °C,0.4525 °C,0.62%,-0.6928,13.0847,69.0567 °C,70.8667 °C,"179,924",4.957021%


In [ ]:
## 7.3 Central Tendency Analysis

This section compares the **mean**, **median**, and **mode** of all operational variables to identify their typical operating values and overall behaviour during the durability experiment. Rather than analysing each variable individually, the objective is to compare the variables collectively in order to distinguish between highly regulated parameters and those that vary under changing operating conditions. The findings provide an overall understanding of the normal operating characteristics of the PEM fuel cell and establish a foundation for the subsequent dispersion, distribution, and degradation analyses.

In [62]:
# ============================================================
# Comparative Central Tendency Summary
# ============================================================

profiles = {
    "Current": current_profile,
    "Voltage": voltage_profile,
    "Power": power_profile,
    "Pressure Anode Inlet": pressure_anode_inlet_profile,
    "Pressure Anode Outlet": pressure_anode_outlet_profile,
    "Pressure Cathode Inlet": pressure_cathode_inlet_profile,
    "Pressure Cathode Outlet": pressure_cathode_outlet_profile,
    "Temperature Anode Endplate": temp_anode_endplate_profile,
    "Temperature Anode Dewpoint": temp_anode_dewpoint_water_profile,
    "Temperature Anode Inlet": temp_anode_inlet_profile,
    "Temperature Anode Outlet": temp_anode_outlet_profile,
    "Temperature Cathode Dewpoint": temp_cathode_dewpoint_water_profile,
    "Temperature Cathode Inlet": temp_cathode_inlet_profile,
    "Temperature Cathode Outlet": temp_cathode_outlet_profile,
    "Total Anode Stack Flow": total_anode_stack_flow_profile,
    "Total Cathode Stack Flow": total_cathode_stack_flow_profile,
}

rows = []

for variable, profile in profiles.items():

    stats = profile["statistics"]

    mean = stats.loc[stats["Metric"] == "Mean", "Value"].iloc[0]
    median = stats.loc[stats["Metric"] == "Median", "Value"].iloc[0]
    mode = stats.loc[stats["Metric"] == "Mode", "Value"].iloc[0]

    rows.append({
        "Variable": variable,
        "Mean": mean,
        "Median": median,
        "Mode": mode
    })

central_tendency_summary = pd.DataFrame(rows)

central_tendency_summary

,Variable,Mean,Median,Mode
0,Current,9.9364 A,9.4848 A,1.7604 A
1,Voltage,0.7567 V,0.7511 V,0.8409 V
2,Power,6.6330 W,6.9300 W,1.4800 W
3,Pressure Anode Inlet,109.9550 kPaG,109.9013 kPaG,109.9013 kPaG
4,Pressure Anode Outlet,110.1856 kPaG,110.3273 kPaG,110.4284 kPaG
5,Pressure Cathode Inlet,109.8189 kPaG,109.8001 kPaG,109.5978 kPaG
6,Pressure Cathode Outlet,107.7650 kPaG,108.3870 kPaG,108.7921 kPaG
7,Temperature Anode Endplate,83.4078 °C,83.5750 °C,83.6715 °C
8,Temperature Anode Dewpoint,54.9523 °C,54.9929 °C,54.9998 °C
9,Temperature Anode Inlet,69.9828 °C,69.9372 °C,69.8949 °C


In [63]:
central_tendency_summary.style.hide(axis="index")

Variable,Mean,Median,Mode
Current,9.9364 A,9.4848 A,1.7604 A
Voltage,0.7567 V,0.7511 V,0.8409 V
Power,6.6330 W,6.9300 W,1.4800 W
Pressure Anode Inlet,109.9550 kPaG,109.9013 kPaG,109.9013 kPaG
Pressure Anode Outlet,110.1856 kPaG,110.3273 kPaG,110.4284 kPaG
Pressure Cathode Inlet,109.8189 kPaG,109.8001 kPaG,109.5978 kPaG
Pressure Cathode Outlet,107.7650 kPaG,108.3870 kPaG,108.7921 kPaG
Temperature Anode Endplate,83.4078 °C,83.5750 °C,83.6715 °C
Temperature Anode Dewpoint,54.9523 °C,54.9929 °C,54.9998 °C
Temperature Anode Inlet,69.9828 °C,69.9372 °C,69.8949 °C


In [ ]:
### Comparative Central Tendency Visualisation

While the comparative central tendency table provides precise numerical values, graphical visualisations facilitate the identification of similarities and differences among operational variables. Because the variables represent different physical quantities and are measured using different units, direct comparison on a single chart would be misleading. Therefore, the variables are grouped according to their physical subsystem into electrical, pressure, temperature, and flow categories. This approach enables meaningful comparisons while preserving the scale and interpretation of each measurement.

In [ ]:
#### Electrical Variables

The electrical variables—current, voltage, and power—are compared using their mean, median, and mode values. This comparison helps identify differences in their typical operating conditions and shows how strongly each variable is influenced by changing electrical load.

In [64]:
# ============================================================
# Central Tendency Comparison: Electrical Variables
# Three Separate Subplots
# ============================================================

electrical = central_tendency_summary[
    central_tendency_summary["Variable"].isin([
        "Current",
        "Voltage",
        "Power"
    ])
].copy()

# ------------------------------------------------------------
# Convert formatted strings to numeric values
# ------------------------------------------------------------

for column in ["Mean", "Median", "Mode"]:
    electrical[column] = (
        electrical[column]
        .str.extract(r"([-+]?\d*\.?\d+)", expand=False)
        .astype(float)
    )

# ------------------------------------------------------------
# Units
# ------------------------------------------------------------

units = {
    "Current": "A",
    "Voltage": "V",
    "Power": "W"
}

# ------------------------------------------------------------
# Consistent colours
# ------------------------------------------------------------

colours = [
    "steelblue",     # Mean
    "darkorange",    # Median
    "forestgreen"    # Mode
]

measures = ["Mean", "Median", "Mode"]

# ------------------------------------------------------------
# Create subplots
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(13,5),
    sharex=False
)

for ax, (_, row) in zip(axes, electrical.iterrows()):

    variable = row["Variable"]

    values = [
        row["Mean"],
        row["Median"],
        row["Mode"]
    ]

    bars = ax.bar(
        measures,
        values,
        color=colours,
        edgecolor="black",
        linewidth=0.8
    )

    ax.set_title(
        variable,
        fontsize=13,
        fontweight="bold"
    )

    ax.set_ylabel(f"{variable} ({units[variable]})")

    ax.grid(
        axis="y",
        alpha=0.3,
        linestyle="--"
    )

    # Value labels

    for bar in bars:

        height = bar.get_height()

        ax.text(
            bar.get_x() + bar.get_width()/2,
            height,
            f"{height:.4f}",
            ha="center",
            va="bottom",
            fontsize=10
        )

# ------------------------------------------------------------
# Shared title
# ------------------------------------------------------------

fig.suptitle(
    "Comparison of Central Tendency Measures for Electrical Variables",
    fontsize=16,
    fontweight="bold"
)

# ------------------------------------------------------------
# Shared legend
# ------------------------------------------------------------

legend_handles = [
    plt.Rectangle((0,0),1,1,color=colours[0]),
    plt.Rectangle((0,0),1,1,color=colours[1]),
    plt.Rectangle((0,0),1,1,color=colours[2]),
]

fig.legend(
    legend_handles,
    measures,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5,-0.02)
)

plt.tight_layout(rect=[0,0.06,1,0.95])

plt.show()

<Figure size 1560x600 with 3 Axes>

In [ ]:
#### Pressure Variables

The pressure variables include the anode and cathode inlet and outlet pressures. Their mean, median, and mode values are compared to assess the stability of pressure regulation across the PEM fuel cell system.

In [65]:
# ============================================================
# Central Tendency Comparison: Pressure Variables
# Four Separate Subplots
# ============================================================

pressure = central_tendency_summary[
    central_tendency_summary["Variable"].str.contains(
        "Pressure",
        case=False,
        na=False
    )
].copy()

# ------------------------------------------------------------
# Convert formatted strings to numeric values
# ------------------------------------------------------------

for column in ["Mean", "Median", "Mode"]:
    pressure[column] = (
        pressure[column]
        .str.extract(r"([-+]?\d*\.?\d+)", expand=False)
        .astype(float)
    )

# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------

measures = ["Mean", "Median", "Mode"]

colours = [
    "steelblue",
    "darkorange",
    "forestgreen"
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 8)
)

axes = axes.flatten()

for ax, (_, row) in zip(axes, pressure.iterrows()):

    variable = row["Variable"]

    values = [
        row["Mean"],
        row["Median"],
        row["Mode"]
    ]

    bars = ax.bar(
        measures,
        values,
        color=colours,
        edgecolor="black",
        linewidth=0.8
    )

    short_title = variable.replace("Pressure ", "")

    ax.set_title(
        short_title,
        fontsize=12,
        fontweight="bold"
    )

    ax.set_ylabel("Pressure (kPaG)")

    ax.grid(
        axis="y",
        alpha=0.3,
        linestyle="--"
    )

    for bar in bars:

        height = bar.get_height()

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"{height:.4f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

fig.suptitle(
    "Central Tendency Comparison of Pressure Variables",
    fontsize=16,
    fontweight="bold"
)

legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=colours[0]),
    plt.Rectangle((0, 0), 1, 1, color=colours[1]),
    plt.Rectangle((0, 0), 1, 1, color=colours[2])
]

fig.legend(
    legend_handles,
    measures,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01)
)

plt.tight_layout(rect=[0, 0.06, 1, 0.95])

plt.show()

<Figure size 1440x960 with 4 Axes>

In [ ]:
#### Temperature Variables

The temperature variables include the anode and cathode endplate, dew point, inlet, and outlet temperatures. Their mean, median, and mode values are compared to examine the stability of the PEM fuel cell thermal management system and to identify differences between the various temperature measurements.

In [66]:
# ============================================================
# Central Tendency Comparison: Temperature Variables
# ============================================================

temperature = central_tendency_summary[
    central_tendency_summary["Variable"].str.contains(
        "Temperature",
        case=False,
        na=False
    )
].copy()

# ------------------------------------------------------------
# Convert formatted strings to numeric values
# ------------------------------------------------------------

for column in ["Mean", "Median", "Mode"]:
    temperature[column] = (
        temperature[column]
        .str.extract(r"([-+]?\d*\.?\d+)", expand=False)
        .astype(float)
    )

# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------

measures = ["Mean", "Median", "Mode"]

colours = [
    "steelblue",
    "darkorange",
    "forestgreen"
]

units = {
    "Temperature Anode Endplate": "°C",
    "Temperature Anode Dewpoint": "°C",
    "Temperature Anode Inlet": "°C",
    "Temperature Anode Outlet": "°C",
    "Temperature Cathode Dewpoint": "°C",
    "Temperature Cathode Inlet": "°C",
    "Temperature Cathode Outlet": "°C"
}

# ------------------------------------------------------------
# Create subplots
# ------------------------------------------------------------

fig, axes = plt.subplots(
    3,
    3,
    figsize=(15, 12)
)

axes = axes.flatten()

for i, (_, row) in enumerate(temperature.iterrows()):

    ax = axes[i]

    variable = row["Variable"]

    values = [
        row["Mean"],
        row["Median"],
        row["Mode"]
    ]

    bars = ax.bar(
        measures,
        values,
        color=colours,
        edgecolor="black",
        linewidth=0.8
    )

    short_title = (
        variable
        .replace("Temperature ", "")
        .replace("Dewpoint", "Dew Point")
    )

    ax.set_title(
        short_title,
        fontsize=11,
        fontweight="bold"
    )

    ax.set_ylabel(f"Temperature ({units[variable]})")

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )

    for bar in bars:

        height = bar.get_height()

        ax.text(
            bar.get_x() + bar.get_width()/2,
            height,
            f"{height:.4f}",
            ha="center",
            va="bottom",
            fontsize=8
        )

# ------------------------------------------------------------
# Hide unused subplot
# ------------------------------------------------------------

axes[-1].axis("off")
axes[-2].axis("off")

# ------------------------------------------------------------
# Figure title
# ------------------------------------------------------------

fig.suptitle(
    "Central Tendency Comparison of Temperature Variables",
    fontsize=17,
    fontweight="bold"
)

# ------------------------------------------------------------
# Shared legend
# ------------------------------------------------------------

legend_handles = [
    plt.Rectangle((0,0),1,1,color=colours[0]),
    plt.Rectangle((0,0),1,1,color=colours[1]),
    plt.Rectangle((0,0),1,1,color=colours[2]),
]

fig.legend(
    legend_handles,
    measures,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5,0.02)
)

plt.tight_layout(rect=[0,0.05,1,0.95])

plt.show()

<Figure size 1800x1440 with 9 Axes>

In [ ]:
#### Reactant Flow Variables

The reactant flow variables represent the total anode and cathode stack flow rates supplied to the PEM fuel cell. Their mean, median, and mode values are compared to assess the typical reactant supply conditions and to identify differences in the distributions of hydrogen and air flow rates.

In [67]:
# ============================================================
# Central Tendency Comparison: Reactant Flow Variables
# ============================================================

flow = central_tendency_summary[
    central_tendency_summary["Variable"].str.contains(
        "Flow",
        case=False,
        na=False
    )
].copy()

# ------------------------------------------------------------
# Convert formatted strings to numeric values
# ------------------------------------------------------------

for column in ["Mean", "Median", "Mode"]:
    flow[column] = (
        flow[column]
        .str.extract(r"([-+]?\d*\.?\d+)", expand=False)
        .astype(float)
    )

# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------

measures = ["Mean", "Median", "Mode"]

colours = [
    "steelblue",
    "darkorange",
    "forestgreen"
]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

for ax, (_, row) in zip(axes, flow.iterrows()):

    values = [
        row["Mean"],
        row["Median"],
        row["Mode"]
    ]

    bars = ax.bar(
        measures,
        values,
        color=colours,
        edgecolor="black",
        linewidth=0.8
    )

    title = (
        row["Variable"]
        .replace("Total ", "")
        .replace(" Stack Flow", "")
    )

    ax.set_title(
        title,
        fontsize=13,
        fontweight="bold"
    )

    ax.set_ylabel("Flow Rate (L/min)")

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )

    for bar in bars:

        height = bar.get_height()

        ax.text(
            bar.get_x() + bar.get_width()/2,
            height,
            f"{height:.4f}",
            ha="center",
            va="bottom",
            fontsize=10
        )

fig.suptitle(
    "Central Tendency Comparison of Reactant Flow Variables",
    fontsize=16,
    fontweight="bold"
)

legend_handles = [
    plt.Rectangle((0,0),1,1,color=colours[0]),
    plt.Rectangle((0,0),1,1,color=colours[1]),
    plt.Rectangle((0,0),1,1,color=colours[2]),
]

fig.legend(
    legend_handles,
    measures,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5,0.02)
)

plt.tight_layout(rect=[0,0.06,1,0.94])

plt.show()

<Figure size 1440x600 with 2 Axes>

In [ ]:
## 7.5 Dispersion Analysis

While central tendency measures describe the typical operating value of each variable, they do not indicate how widely the observations are distributed around that value. Dispersion measures quantify the variability within the dataset and provide insight into the consistency and stability of the PEM fuel cell operating conditions.

This section compares the variability of all operational variables using the range, variance, standard deviation, interquartile range (IQR), and coefficient of variation (CV). Together, these measures identify variables that remain tightly regulated throughout the durability experiment and those that exhibit substantial variation due to changing operating conditions.

In [68]:
# ============================================================
# Dispersion Summary Table
# ============================================================

dispersion_summary = statistics_summary[
    [
        "Variable",
        "Range",
        "Variance",
        "Standard Deviation",
        "IQR",
        "Coefficient of Variation (%)"
    ]
].copy()

display(dispersion_summary)

,Variable,Range,Variance,Standard Deviation,IQR,Coefficient of Variation (%)
0,Current,35.5439 A,88.210841 A²,9.3921 A,13.0559 A,94.52%
1,Voltage,1.0085 V,0.009453 V²,0.0972 V,0.1557 V,12.85%
2,Power,22.6500 W,29.867930 W²,5.4652 W,8.8400 W,82.39%
3,Pressure Anode Inlet,77.7277 kPaG,1.533169 kPaG²,1.2382 kPaG,0.3035 kPaG,1.13%
4,Pressure Anode Outlet,76.7615 kPaG,1.598050 kPaG²,1.2641 kPaG,0.4044 kPaG,1.15%
5,Pressure Cathode Inlet,27.3299 kPaG,2.221347 kPaG²,1.4904 kPaG,1.1129 kPaG,1.36%
6,Pressure Cathode Outlet,12.0641 kPaG,2.550717 kPaG²,1.5971 kPaG,1.7216 kPaG,1.48%
7,Temperature Anode Endplate,3.5722 °C,0.306581 °C²,0.5537 °C,0.8549 °C,0.66%
8,Temperature Anode Dewpoint,4.1853 °C,0.043535 °C²,0.2086 °C,0.0841 °C,0.38%
9,Temperature Anode Inlet,13.6280 °C,0.190779 °C²,0.4368 °C,0.4525 °C,0.62%


In [ ]:
### 7.5.2 Comparative Dispersion Visualisations

To ensure meaningful comparison, dispersion is examined separately for the electrical, pressure, temperature, and reactant-flow subsystems. The coefficient of variation is used as the principal comparative measure because it expresses variability relative to the mean and is therefore suitable for comparing variables with different units and magnitudes.

In [69]:
# ============================================================
# Electrical Variable Dispersion
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd

electrical_dispersion = dispersion_summary[
    dispersion_summary["Variable"].isin(
        ["Current", "Voltage", "Power"]
    )
].copy()

# Convert formatted CV values such as "94.52%" to numeric values
electrical_dispersion["CV_numeric"] = (
    electrical_dispersion["Coefficient of Variation (%)"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .astype(float)
)

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    electrical_dispersion["Variable"],
    electrical_dispersion["CV_numeric"],
    edgecolor="black"
)

ax.set_title(
    "Relative Dispersion of Electrical Variables",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("Electrical Variable")
ax.set_ylabel("Coefficient of Variation (%)")
ax.grid(axis="y", linestyle="--", alpha=0.5)

for bar, value in zip(bars, electrical_dispersion["CV_numeric"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
        fontsize=10
    )

ax.set_ylim(
    0,
    electrical_dispersion["CV_numeric"].max() * 1.15
)

plt.tight_layout()
plt.show()

<Figure size 1080x600 with 1 Axes>

In [70]:
# ============================================================
# Pressure Variable Dispersion
# ============================================================

import matplotlib.pyplot as plt

pressure_dispersion = dispersion_summary[
    dispersion_summary["Variable"].isin(
        [
            "Pressure Anode Inlet",
            "Pressure Anode Outlet",
            "Pressure Cathode Inlet",
            "Pressure Cathode Outlet",
        ]
    )
].copy()

pressure_dispersion["CV_numeric"] = (
    pressure_dispersion["Coefficient of Variation (%)"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .astype(float)
)

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(
    pressure_dispersion["Variable"],
    pressure_dispersion["CV_numeric"],
    edgecolor="black"
)

ax.set_title(
    "Relative Dispersion of Pressure Variables",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Pressure Variable")
ax.set_ylabel("Coefficient of Variation (%)")

ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.xticks(rotation=15)

for bar, value in zip(bars, pressure_dispersion["CV_numeric"]):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.03,
        f"{value:.2f}%",
        ha="center",
        fontsize=10
    )

ax.set_ylim(0, pressure_dispersion["CV_numeric"].max()*1.25)

plt.tight_layout()
plt.show()

<Figure size 1200x600 with 1 Axes>

In [71]:
# ============================================================
# Temperature Variable Dispersion
# ============================================================

import matplotlib.pyplot as plt

temperature_dispersion = dispersion_summary[
    dispersion_summary["Variable"].isin(
        [
            "Temperature Anode Endplate",
            "Temperature Anode Dewpoint",
            "Temperature Anode Inlet",
            "Temperature Anode Outlet",
            "Temperature Cathode Dewpoint",
            "Temperature Cathode Inlet",
            "Temperature Cathode Outlet",
        ]
    )
].copy()

temperature_dispersion["CV_numeric"] = (
    temperature_dispersion["Coefficient of Variation (%)"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .astype(float)
)

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(
    temperature_dispersion["Variable"],
    temperature_dispersion["CV_numeric"],
    edgecolor="black"
)

ax.set_title(
    "Relative Dispersion of Temperature Variables",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Temperature Variable")
ax.set_ylabel("Coefficient of Variation (%)")

ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.xticks(rotation=20, ha="right")

for bar, value in zip(bars, temperature_dispersion["CV_numeric"]):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.08,
        f"{value:.2f}%",
        ha="center",
        fontsize=9
    )

ax.set_ylim(0, temperature_dispersion["CV_numeric"].max()*1.20)

plt.tight_layout()
plt.show()

<Figure size 1440x720 with 1 Axes>

In [72]:
# ============================================================
# Reactant Flow Variable Dispersion
# ============================================================

import matplotlib.pyplot as plt

flow_dispersion = dispersion_summary[
    dispersion_summary["Variable"].isin(
        [
            "Total Anode Stack Flow",
            "Total Cathode Stack Flow",
        ]
    )
].copy()

flow_dispersion["CV_numeric"] = (
    flow_dispersion["Coefficient of Variation (%)"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .astype(float)
)

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    flow_dispersion["Variable"],
    flow_dispersion["CV_numeric"],
    edgecolor="black"
)

ax.set_title(
    "Relative Dispersion of Reactant Flow Variables",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Reactant Flow Variable")
ax.set_ylabel("Coefficient of Variation (%)")

ax.grid(axis="y", linestyle="--", alpha=0.5)

for bar, value in zip(bars, flow_dispersion["CV_numeric"]):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        fontsize=10
    )

ax.set_ylim(0, flow_dispersion["CV_numeric"].max() * 1.15)

plt.tight_layout()
plt.show()

<Figure size 960x600 with 1 Axes>

In [ ]:
## 7.6 Distribution Analysis

While dispersion measures quantify the magnitude of variability, distribution analysis examines how observations are distributed across the range of each variable. Understanding the distribution of operational measurements is essential for identifying common operating regions, dominant operating conditions, and deviations from normal behaviour.

This section compares the distribution characteristics of the electrical, pressure, temperature, and reactant flow variables using the histograms generated during the variable profile analysis. The comparative assessment focuses on the concentration of observations, distribution symmetry, spread, and the presence of multiple operating regions that may reflect different stages or operating modes of the PEM fuel cell throughout the durability experiment.

In [ ]:
7.6.1 Comparative Distribution Visualisations 
There is no need to generate new plots because we already created high-quality histograms for each variable during the Variable Profile Analysis.
We'll simply reuse those figures in the dissertation and write a comparative interpretation based on them.

In [ ]:
## 7.7 Distribution Shape Analysis

While distribution analysis provides a visual understanding of how observations are distributed, distribution shape analysis quantitatively describes the symmetry and peakedness of each variable using skewness and kurtosis. These statistical measures help identify deviations from a normal distribution, reveal the presence of long tails or extreme observations, and provide insight into the operational behaviour of the PEM fuel cell variables.

This section compares the distribution shape of the electrical, pressure, temperature, and reactant flow variables to determine which operational parameters exhibit approximately symmetric behaviour and which demonstrate pronounced asymmetry or heavy-tailed characteristics.

In [73]:
# ============================================================
# Distribution Shape Summary Table
# ============================================================

shape_summary = statistics_summary[
    [
        "Variable",
        "Skewness",
        "Kurtosis"
    ]
].copy()

display(shape_summary)

,Variable,Skewness,Kurtosis
0,Current,1.0395,0.3472
1,Voltage,-0.4867,-0.6132
2,Power,0.6275,-0.7118
3,Pressure Anode Inlet,-23.9786,830.5370
4,Pressure Anode Outlet,-21.9322,745.6754
5,Pressure Cathode Inlet,-1.8667,6.3340
6,Pressure Cathode Outlet,-1.7283,2.6537
7,Temperature Anode Endplate,-0.4316,-0.5413
8,Temperature Anode Dewpoint,-2.8794,14.0682
9,Temperature Anode Inlet,-0.6928,13.0847


In [74]:
# ============================================================
# Electrical Variable Distribution Shape
# ============================================================
# ------------------------------------------------------------
# Extract Electrical Variables
# ------------------------------------------------------------

electrical_shape = shape_summary[
    shape_summary["Variable"].isin(
        [
            "Current",
            "Voltage",
            "Power"
        ]
    )
].copy()

# ------------------------------------------------------------
# Convert statistical columns to numeric
# ------------------------------------------------------------

electrical_shape["Skewness"] = pd.to_numeric(
    electrical_shape["Skewness"],
    errors="coerce"
)

electrical_shape["Kurtosis"] = pd.to_numeric(
    electrical_shape["Kurtosis"],
    errors="coerce"
)

# Check that conversion did not create missing values
if electrical_shape[["Skewness", "Kurtosis"]].isna().any().any():
    raise ValueError(
        "Skewness or Kurtosis contains values that could not be converted "
        "to numeric format."
    )

# ------------------------------------------------------------
# Preserve the intended variable order
# ------------------------------------------------------------

variable_order = [
    "Current",
    "Voltage",
    "Power"
]

electrical_shape["Variable"] = pd.Categorical(
    electrical_shape["Variable"],
    categories=variable_order,
    ordered=True
)

electrical_shape = (
    electrical_shape
    .sort_values("Variable")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Assign colours according to direction
# Positive = blue, Negative = red
# ------------------------------------------------------------

skewness_colours = [
    "steelblue" if value >= 0 else "indianred"
    for value in electrical_shape["Skewness"]
]

kurtosis_colours = [
    "steelblue" if value >= 0 else "indianred"
    for value in electrical_shape["Kurtosis"]
]

# ------------------------------------------------------------
# Create Figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(13, 5)
)

# ============================================================
# Skewness Plot
# ============================================================

bars1 = axes[0].barh(
    electrical_shape["Variable"],
    electrical_shape["Skewness"],
    color=skewness_colours,
    edgecolor="black",
    linewidth=1
)

axes[0].set_title(
    "Skewness of Electrical Variables",
    fontsize=13,
    fontweight="bold"
)

axes[0].set_xlabel("Skewness")
axes[0].set_ylabel("Electrical Variable")

axes[0].axvline(
    x=0,
    color="black",
    linewidth=1.2
)

axes[0].grid(
    axis="x",
    linestyle="--",
    alpha=0.5
)

axes[0].set_axisbelow(True)

# Dynamic skewness limits
skew_min = electrical_shape["Skewness"].min()
skew_max = electrical_shape["Skewness"].max()
skew_span = max(skew_max - skew_min, 1)

axes[0].set_xlim(
    min(skew_min - 0.15 * skew_span, -0.10),
    max(skew_max + 0.15 * skew_span, 0.10)
)

# Add centred values inside bars
for bar, value in zip(
    bars1,
    electrical_shape["Skewness"]
):
    axes[0].text(
        value / 2,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}",
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
        color="white"
    )

# ============================================================
# Kurtosis Plot
# ============================================================

bars2 = axes[1].barh(
    electrical_shape["Variable"],
    electrical_shape["Kurtosis"],
    color=kurtosis_colours,
    edgecolor="black",
    linewidth=1
)

axes[1].set_title(
    "Kurtosis of Electrical Variables",
    fontsize=13,
    fontweight="bold"
)

axes[1].set_xlabel("Excess Kurtosis")
axes[1].set_ylabel("Electrical Variable")

axes[1].axvline(
    x=0,
    color="black",
    linewidth=1.2
)

axes[1].grid(
    axis="x",
    linestyle="--",
    alpha=0.5
)

axes[1].set_axisbelow(True)

# Dynamic kurtosis limits
kurt_min = electrical_shape["Kurtosis"].min()
kurt_max = electrical_shape["Kurtosis"].max()
kurt_span = max(kurt_max - kurt_min, 1)

axes[1].set_xlim(
    min(kurt_min - 0.15 * kurt_span, -0.10),
    max(kurt_max + 0.15 * kurt_span, 0.10)
)

# Add centred values inside bars
for bar, value in zip(
    bars2,
    electrical_shape["Kurtosis"]
):
    axes[1].text(
        value / 2,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}",
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
        color="white"
    )

# ------------------------------------------------------------
# Overall Figure Formatting
# ------------------------------------------------------------

fig.suptitle(
    "Distribution Shape of Electrical Variables",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, 0.93],
    w_pad=3
)

plt.show()

<Figure size 1560x600 with 2 Axes>

In [75]:
# ============================================================
# Pressure Variable Distribution Shape
# ============================================================

# ------------------------------------------------------------
# Extract Pressure Variables
# ------------------------------------------------------------

pressure_shape = shape_summary[
    shape_summary["Variable"].isin(
        [
            "Pressure Anode Inlet",
            "Pressure Anode Outlet",
            "Pressure Cathode Inlet",
            "Pressure Cathode Outlet"
        ]
    )
].copy()

# ------------------------------------------------------------
# Convert statistical columns to numeric
# ------------------------------------------------------------

pressure_shape["Skewness"] = pd.to_numeric(
    pressure_shape["Skewness"],
    errors="coerce"
)

pressure_shape["Kurtosis"] = pd.to_numeric(
    pressure_shape["Kurtosis"],
    errors="coerce"
)

# Validate conversion
if pressure_shape[["Skewness", "Kurtosis"]].isna().any().any():
    raise ValueError(
        "Skewness or Kurtosis contains values that could not "
        "be converted to numeric format."
    )

# A logarithmic axis requires strictly positive values
if (pressure_shape["Kurtosis"] <= 0).any():
    raise ValueError(
        "The logarithmic kurtosis plot requires all excess "
        "kurtosis values to be greater than zero."
    )

# ------------------------------------------------------------
# Preserve intended variable order
# ------------------------------------------------------------

variable_order = [
    "Pressure Anode Inlet",
    "Pressure Anode Outlet",
    "Pressure Cathode Inlet",
    "Pressure Cathode Outlet"
]

pressure_shape["Variable"] = pd.Categorical(
    pressure_shape["Variable"],
    categories=variable_order,
    ordered=True
)

pressure_shape = (
    pressure_shape
    .sort_values("Variable")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Assign colours according to direction
# Positive = blue, Negative = red
# ------------------------------------------------------------

skewness_colours = [
    "steelblue" if value >= 0 else "indianred"
    for value in pressure_shape["Skewness"]
]

# All pressure kurtosis values are positive
kurtosis_colours = [
    "steelblue" if value >= 0 else "indianred"
    for value in pressure_shape["Kurtosis"]
]

# ------------------------------------------------------------
# Create Figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(15, 6)
)

# ============================================================
# Skewness Plot
# ============================================================

bars1 = axes[0].barh(
    pressure_shape["Variable"],
    pressure_shape["Skewness"],
    color=skewness_colours,
    edgecolor="black",
    linewidth=1
)

axes[0].set_title(
    "Skewness of Pressure Variables",
    fontsize=13,
    fontweight="bold"
)

axes[0].set_xlabel("Skewness")
axes[0].set_ylabel("Pressure Variable")

axes[0].axvline(
    x=0,
    color="black",
    linewidth=1.2
)

axes[0].grid(
    axis="x",
    linestyle="--",
    alpha=0.5
)

axes[0].set_axisbelow(True)

# Dynamic skewness limits
skew_min = pressure_shape["Skewness"].min()
skew_max = pressure_shape["Skewness"].max()
skew_span = max(skew_max - skew_min, 1)

axes[0].set_xlim(
    min(skew_min - 0.15 * skew_span, -0.10),
    max(skew_max + 0.15 * skew_span, 0.10)
)

# Centred value labels
for bar, value in zip(
    bars1,
    pressure_shape["Skewness"]
):
    axes[0].text(
        value / 2,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}",
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
        color="white"
    )

# ============================================================
# Kurtosis Plot
# ============================================================

bars2 = axes[1].barh(
    pressure_shape["Variable"],
    pressure_shape["Kurtosis"],
    color=kurtosis_colours,
    edgecolor="black",
    linewidth=1
)

axes[1].set_title(
    "Excess Kurtosis of Pressure Variables",
    fontsize=13,
    fontweight="bold"
)

axes[1].set_xlabel("Excess Kurtosis — Logarithmic Scale")
axes[1].set_ylabel("Pressure Variable")

# Apply logarithmic x-axis
axes[1].set_xscale("log")

axes[1].grid(
    axis="x",
    which="both",
    linestyle="--",
    alpha=0.5
)

axes[1].set_axisbelow(True)

# Dynamic logarithmic limits
kurt_min = pressure_shape["Kurtosis"].min()
kurt_max = pressure_shape["Kurtosis"].max()

axes[1].set_xlim(
    kurt_min * 0.60,
    kurt_max * 1.50
)

# Place labels at the end of each logarithmic bar
for bar, value in zip(
    bars2,
    pressure_shape["Kurtosis"]
):
    axes[1].text(
        value * 1.05,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}",
        ha="left",
        va="center",
        fontsize=9,
        fontweight="bold"
    )

# ------------------------------------------------------------
# Overall Figure Formatting
# ------------------------------------------------------------

fig.suptitle(
    "Distribution Shape of Pressure Variables",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, 0.93],
    w_pad=4
)

plt.show()

<Figure size 1800x720 with 2 Axes>

In [76]:
# ============================================================
# Temperature Variable Distribution Shape
# ============================================================

# ------------------------------------------------------------
# Extract Temperature Variables
# ------------------------------------------------------------

temperature_shape = shape_summary[
    shape_summary["Variable"].isin(
        [
            "Temperature Anode Endplate",
            "Temperature Anode Dewpoint",
            "Temperature Anode Inlet",
            "Temperature Anode Outlet",
            "Temperature Cathode Dewpoint",
            "Temperature Cathode Inlet",
            "Temperature Cathode Outlet"
        ]
    )
].copy()

# ------------------------------------------------------------
# Convert statistical columns to numeric
# ------------------------------------------------------------

temperature_shape["Skewness"] = pd.to_numeric(
    temperature_shape["Skewness"],
    errors="coerce"
)

temperature_shape["Excess Kurtosis"] = pd.to_numeric(
    temperature_shape["Kurtosis"],
    errors="coerce"
)

# Validate numeric conversion
if temperature_shape[
    ["Skewness", "Excess Kurtosis"]
].isna().any().any():
    raise ValueError(
        "Skewness or Kurtosis contains values that could not "
        "be converted to numeric format."
    )

# ------------------------------------------------------------
# Preserve intended variable order
# ------------------------------------------------------------

variable_order = [
    "Temperature Anode Endplate",
    "Temperature Anode Dewpoint",
    "Temperature Anode Inlet",
    "Temperature Anode Outlet",
    "Temperature Cathode Dewpoint",
    "Temperature Cathode Inlet",
    "Temperature Cathode Outlet"
]

temperature_shape["Variable"] = pd.Categorical(
    temperature_shape["Variable"],
    categories=variable_order,
    ordered=True
)

temperature_shape = (
    temperature_shape
    .sort_values("Variable")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Assign colours according to direction
# Positive = blue, Negative = red
# ------------------------------------------------------------

skewness_colours = [
    "steelblue" if value >= 0 else "indianred"
    for value in temperature_shape["Skewness"]
]

kurtosis_colours = [
    "steelblue" if value >= 0 else "indianred"
    for value in temperature_shape["Excess Kurtosis"]
]

# ------------------------------------------------------------
# Create Figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(16, 7)
)

# ============================================================
# Skewness Plot
# ============================================================

bars1 = axes[0].barh(
    temperature_shape["Variable"],
    temperature_shape["Skewness"],
    color=skewness_colours,
    edgecolor="black",
    linewidth=1
)

axes[0].set_title(
    "Skewness of Temperature Variables",
    fontsize=13,
    fontweight="bold"
)

axes[0].set_xlabel("Skewness")
axes[0].set_ylabel("Temperature Variable")

axes[0].axvline(
    x=0,
    color="black",
    linewidth=1.2
)

axes[0].grid(
    axis="x",
    linestyle="--",
    alpha=0.5
)

axes[0].set_axisbelow(True)

# Dynamic skewness axis limits
skew_min = temperature_shape["Skewness"].min()
skew_max = temperature_shape["Skewness"].max()
skew_span = max(skew_max - skew_min, 1)

skew_left = min(
    skew_min - 0.18 * skew_span,
    -0.10
)

skew_right = max(
    skew_max + 0.18 * skew_span,
    0.10
)

axes[0].set_xlim(skew_left, skew_right)

# Total displayed axis width
skew_axis_width = skew_right - skew_left

# Add adaptive labels:
# large bars -> inside, white
# small bars -> outside, black
for bar, value in zip(
    bars1,
    temperature_shape["Skewness"]
):
    relative_length = abs(value) / skew_axis_width

    if relative_length >= 0.10:
        # Label inside the bar
        axes[0].text(
            value / 2,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
            color="white"
        )
    else:
        # Label outside the bar
        offset = 0.015 * skew_axis_width

        axes[0].text(
            value + offset if value >= 0 else value - offset,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.2f}",
            ha="left" if value >= 0 else "right",
            va="center",
            fontsize=9,
            fontweight="bold",
            color="black"
        )

# ============================================================
# Excess Kurtosis Plot
# ============================================================

bars2 = axes[1].barh(
    temperature_shape["Variable"],
    temperature_shape["Excess Kurtosis"],
    color=kurtosis_colours,
    edgecolor="black",
    linewidth=1
)

axes[1].set_title(
    "Excess Kurtosis of Temperature Variables",
    fontsize=13,
    fontweight="bold"
)

axes[1].set_xlabel("Excess Kurtosis")
axes[1].set_ylabel("Temperature Variable")

axes[1].axvline(
    x=0,
    color="black",
    linewidth=1.2
)

axes[1].grid(
    axis="x",
    linestyle="--",
    alpha=0.5
)

axes[1].set_axisbelow(True)

# Dynamic excess-kurtosis axis limits
kurt_min = temperature_shape["Excess Kurtosis"].min()
kurt_max = temperature_shape["Excess Kurtosis"].max()
kurt_span = max(kurt_max - kurt_min, 1)

kurt_left = min(
    kurt_min - 0.18 * kurt_span,
    -0.10
)

kurt_right = max(
    kurt_max + 0.18 * kurt_span,
    0.10
)

axes[1].set_xlim(kurt_left, kurt_right)

# Total displayed axis width
kurt_axis_width = kurt_right - kurt_left

# Add adaptive labels:
# large bars -> inside, white
# small bars -> outside, black
for bar, value in zip(
    bars2,
    temperature_shape["Excess Kurtosis"]
):
    relative_length = abs(value) / kurt_axis_width

    if relative_length >= 0.10:
        # Label inside the bar
        axes[1].text(
            value / 2,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
            color="white"
        )
    else:
        # Label outside the bar
        offset = 0.015 * kurt_axis_width

        axes[1].text(
            value + offset if value >= 0 else value - offset,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.2f}",
            ha="left" if value >= 0 else "right",
            va="center",
            fontsize=9,
            fontweight="bold",
            color="black"
        )

# ------------------------------------------------------------
# Overall Figure Formatting
# ------------------------------------------------------------

fig.suptitle(
    "Distribution Shape of Temperature Variables",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0, 0, 1, 0.94],
    w_pad=4
)

plt.show()

<Figure size 1920x840 with 2 Axes>

In [77]:
# ============================================================
# Reactant Flow Variable Distribution Shape
# ============================================================

# ------------------------------------------------------------
# Extract Reactant Flow Variables
# ------------------------------------------------------------

flow_shape = shape_summary[
    shape_summary["Variable"].isin(
        [
            "Total Anode Stack Flow",
            "Total Cathode Stack Flow"
        ]
    )
].copy()

# ------------------------------------------------------------
# Convert to numeric
# ------------------------------------------------------------

flow_shape["Skewness"] = pd.to_numeric(
    flow_shape["Skewness"],
    errors="coerce"
)

flow_shape["Excess Kurtosis"] = pd.to_numeric(
    flow_shape["Kurtosis"],
    errors="coerce"
)

# ------------------------------------------------------------
# Preserve variable order
# ------------------------------------------------------------

variable_order = [
    "Total Anode Stack Flow",
    "Total Cathode Stack Flow"
]

flow_shape["Variable"] = pd.Categorical(
    flow_shape["Variable"],
    categories=variable_order,
    ordered=True
)

flow_shape = (
    flow_shape
    .sort_values("Variable")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Colours
# ------------------------------------------------------------

skew_colours = [
    "steelblue" if value >= 0 else "indianred"
    for value in flow_shape["Skewness"]
]

kurtosis_colours = [
    "steelblue" if value >= 0 else "indianred"
    for value in flow_shape["Excess Kurtosis"]
]

# ------------------------------------------------------------
# Create Figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14,6)
)

# ============================================================
# Skewness
# ============================================================

bars1 = axes[0].barh(
    flow_shape["Variable"],
    flow_shape["Skewness"],
    color=skew_colours,
    edgecolor="black",
    linewidth=1
)

axes[0].set_title(
    "Skewness of Reactant Flow Variables",
    fontsize=13,
    fontweight="bold"
)

axes[0].set_xlabel("Skewness")
axes[0].set_ylabel("Reactant Flow Variable")

axes[0].axvline(
    0,
    color="black",
    linewidth=1.2
)

axes[0].grid(
    axis="x",
    linestyle="--",
    alpha=0.5
)

axes[0].set_axisbelow(True)

skew_max = flow_shape["Skewness"].max()

axes[0].set_xlim(
    -0.2,
    skew_max * 1.25
)

for bar, value in zip(
    bars1,
    flow_shape["Skewness"]
):

    axes[0].text(
        value/2,
        bar.get_y()+bar.get_height()/2,
        f"{value:.2f}",
        ha="center",
        va="center",
        fontsize=10,
        color="white",
        fontweight="bold"
    )

# ============================================================
# Excess Kurtosis
# ============================================================

bars2 = axes[1].barh(
    flow_shape["Variable"],
    flow_shape["Excess Kurtosis"],
    color=kurtosis_colours,
    edgecolor="black",
    linewidth=1
)

axes[1].set_title(
    "Excess Kurtosis of Reactant Flow Variables",
    fontsize=13,
    fontweight="bold"
)

axes[1].set_xlabel("Excess Kurtosis")
axes[1].set_ylabel("Reactant Flow Variable")

axes[1].axvline(
    0,
    color="black",
    linewidth=1.2
)

axes[1].grid(
    axis="x",
    linestyle="--",
    alpha=0.5
)

axes[1].set_axisbelow(True)

kurt_max = flow_shape["Excess Kurtosis"].max()

axes[1].set_xlim(
    -0.2,
    kurt_max * 1.25
)

for bar, value in zip(
    bars2,
    flow_shape["Excess Kurtosis"]
):

    axes[1].text(
        value/2,
        bar.get_y()+bar.get_height()/2,
        f"{value:.2f}",
        ha="center",
        va="center",
        fontsize=10,
        color="white",
        fontweight="bold"
    )

# ------------------------------------------------------------
# Overall formatting
# ------------------------------------------------------------

fig.suptitle(
    "Distribution Shape of Reactant Flow Variables",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout(
    rect=[0,0,1,0.94],
    w_pad=3
)

plt.show()

<Figure size 1680x720 with 2 Axes>

In [ ]:
### 7.7.1 Outlier Assessment

Outlier assessment evaluates observations that fall outside the expected operating range of each variable using the Interquartile Range (IQR) method. Unlike the previous analyses, which examined the central characteristics and distributional behaviour of the variables, this section investigates the occurrence and magnitude of statistically unusual observations.

The objective is to identify which operational variables experience the greatest proportion of extreme values and to determine whether these observations are consistent with the physical behaviour of the PEM fuel cell or indicate transient operating conditions. The findings provide insight into data variability, operational stability, and potential indicators of degradation while informing subsequent feature engineering and machine learning model development.

In [78]:
# ============================================================
# Outlier Summary Table
# ============================================================

outlier_summary = statistics_summary[
    [
        "Variable",
        "IQR Lower Bound",
        "IQR Upper Bound",
        "IQR Outlier Count",
        "IQR Outliers (%)"
    ]
].copy()

display(outlier_summary)

,Variable,IQR Lower Bound,IQR Upper Bound,IQR Outlier Count,IQR Outliers (%)
0,Current,-17.8234 A,34.4001 A,"135,333",3.728511%
1,Voltage,0.4536 V,1.0763 V,4,0.000110%
2,Power,-11.7700 W,23.5900 W,0,0.000000%
3,Pressure Anode Inlet,109.4460 kPaG,110.6601 kPaG,"280,556",7.729497%
4,Pressure Anode Outlet,109.4173 kPaG,111.0350 kPaG,"302,492",8.333848%
5,Pressure Cathode Inlet,107.7260 kPaG,112.1777 kPaG,"315,950",8.704624%
6,Pressure Cathode Outlet,104.4880 kPaG,111.3746 kPaG,"217,194",5.983833%
7,Temperature Anode Endplate,81.6535 °C,85.0732 °C,128,0.003526%
8,Temperature Anode Dewpoint,54.8196 °C,55.1560 °C,"588,599",16.216278%
9,Temperature Anode Inlet,69.0567 °C,70.8667 °C,"179,924",4.957021%


In [ ]:
The table summarises the Interquartile Range (IQR) boundaries together with the number and percentage of observations classified as statistical outliers for each operational variable. These metrics provide an overview of the variables that exhibit the greatest occurrence of extreme observations before subsystem-level comparisons are performed.

In [79]:
# ============================================================
# Electrical Variable Outlier Assessment
# ============================================================

# ------------------------------------------------------------
# Extract Electrical Variables
# ------------------------------------------------------------

electrical_outliers = outlier_summary[
    outlier_summary["Variable"].isin(
        [
            "Current",
            "Voltage",
            "Power"
        ]
    )
].copy()

# ------------------------------------------------------------
# Convert percentage to numeric
# ------------------------------------------------------------

electrical_outliers["Outlier Percentage"] = (
    electrical_outliers["IQR Outliers (%)"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .astype(float)
)

# ------------------------------------------------------------
# Create Figure
# ------------------------------------------------------------

plt.figure(figsize=(8,5))

bars = plt.bar(
    electrical_outliers["Variable"],
    electrical_outliers["Outlier Percentage"],
    edgecolor="black",
    linewidth=1
)

plt.title(
    "IQR Outlier Percentage of Electrical Variables",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Electrical Variable")
plt.ylabel("Outliers (%)")

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.5
)

plt.gca().set_axisbelow(True)

# ------------------------------------------------------------
# Value labels
# ------------------------------------------------------------

max_value = electrical_outliers["Outlier Percentage"].max()

plt.ylim(
    0,
    max_value * 1.15
)

for bar, value in zip(
    bars,
    electrical_outliers["Outlier Percentage"]
):

    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + max_value*0.02,
        f"{value:.2f}%",
        ha="center",
        fontsize=10,
        fontweight="bold"
    )

plt.tight_layout()

plt.show()

<Figure size 960x600 with 1 Axes>

In [80]:
# ============================================================
# Pressure Variable Outlier Assessment
# ============================================================

# ------------------------------------------------------------
# Extract Pressure Variables
# ------------------------------------------------------------

pressure_outliers = outlier_summary[
    outlier_summary["Variable"].isin(
        [
            "Pressure Anode Inlet",
            "Pressure Anode Outlet",
            "Pressure Cathode Inlet",
            "Pressure Cathode Outlet"
        ]
    )
].copy()

# ------------------------------------------------------------
# Convert percentage to numeric
# ------------------------------------------------------------

pressure_outliers["Outlier Percentage"] = (
    pressure_outliers["IQR Outliers (%)"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .astype(float)
)

# ------------------------------------------------------------
# Preserve variable order
# ------------------------------------------------------------

variable_order = [
    "Pressure Anode Inlet",
    "Pressure Anode Outlet",
    "Pressure Cathode Inlet",
    "Pressure Cathode Outlet"
]

pressure_outliers["Variable"] = pd.Categorical(
    pressure_outliers["Variable"],
    categories=variable_order,
    ordered=True
)

pressure_outliers = (
    pressure_outliers
    .sort_values("Variable")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Create Figure
# ------------------------------------------------------------

plt.figure(figsize=(9,5))

bars = plt.bar(
    pressure_outliers["Variable"],
    pressure_outliers["Outlier Percentage"],
    edgecolor="black",
    linewidth=1
)

plt.title(
    "IQR Outlier Percentage of Pressure Variables",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Pressure Variable")
plt.ylabel("Outliers (%)")

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.5
)

plt.gca().set_axisbelow(True)

# ------------------------------------------------------------
# Dynamic axis limits
# ------------------------------------------------------------

max_value = pressure_outliers["Outlier Percentage"].max()

plt.ylim(
    0,
    max_value * 1.15
)

# ------------------------------------------------------------
# Value labels
# ------------------------------------------------------------

for bar, value in zip(
    bars,
    pressure_outliers["Outlier Percentage"]
):

    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + max_value * 0.02,
        f"{value:.2f}%",
        ha="center",
        fontsize=10,
        fontweight="bold"
    )

plt.tight_layout()

plt.show()

<Figure size 1080x600 with 1 Axes>

In [81]:
# ============================================================
# Temperature Variable Outlier Assessment
# ============================================================

# ------------------------------------------------------------
# Extract Temperature Variables
# ------------------------------------------------------------

temperature_outliers = outlier_summary[
    outlier_summary["Variable"].isin(
        [
            "Temperature Anode Endplate",
            "Temperature Anode Dewpoint",
            "Temperature Anode Inlet",
            "Temperature Anode Outlet",
            "Temperature Cathode Dewpoint",
            "Temperature Cathode Inlet",
            "Temperature Cathode Outlet"
        ]
    )
].copy()

# ------------------------------------------------------------
# Convert percentage to numeric
# ------------------------------------------------------------

temperature_outliers["Outlier Percentage"] = (
    temperature_outliers["IQR Outliers (%)"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .astype(float)
)

# ------------------------------------------------------------
# Preserve variable order
# ------------------------------------------------------------

variable_order = [
    "Temperature Anode Endplate",
    "Temperature Anode Dewpoint",
    "Temperature Anode Inlet",
    "Temperature Anode Outlet",
    "Temperature Cathode Dewpoint",
    "Temperature Cathode Inlet",
    "Temperature Cathode Outlet"
]

temperature_outliers["Variable"] = pd.Categorical(
    temperature_outliers["Variable"],
    categories=variable_order,
    ordered=True
)

temperature_outliers = (
    temperature_outliers
    .sort_values("Variable")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Create Figure
# ------------------------------------------------------------

plt.figure(figsize=(12,5))

bars = plt.bar(
    temperature_outliers["Variable"],
    temperature_outliers["Outlier Percentage"],
    edgecolor="black",
    linewidth=1
)

plt.title(
    "IQR Outlier Percentage of Temperature Variables",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Temperature Variable")
plt.ylabel("Outliers (%)")

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.5
)

plt.gca().set_axisbelow(True)

# Rotate labels

plt.xticks(rotation=20, ha="right")

# ------------------------------------------------------------
# Dynamic axis
# ------------------------------------------------------------

max_value = temperature_outliers["Outlier Percentage"].max()

plt.ylim(
    0,
    max_value * 1.15
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

for bar, value in zip(
    bars,
    temperature_outliers["Outlier Percentage"]
):

    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + max_value*0.02,
        f"{value:.2f}%",
        ha="center",
        fontsize=10,
        fontweight="bold"
    )

plt.tight_layout()

plt.show()

<Figure size 1440x600 with 1 Axes>

In [82]:
# ============================================================
# Reactant Flow Variable Outlier Assessment
# ============================================================

# ------------------------------------------------------------
# Extract Reactant Flow Variables
# ------------------------------------------------------------

flow_outliers = outlier_summary[
    outlier_summary["Variable"].isin(
        [
            "Total Anode Stack Flow",
            "Total Cathode Stack Flow"
        ]
    )
].copy()

# ------------------------------------------------------------
# Convert outlier percentages to numeric
# ------------------------------------------------------------

flow_outliers["Outlier Percentage"] = pd.to_numeric(
    flow_outliers["IQR Outliers (%)"]
    .astype(str)
    .str.replace("%", "", regex=False),
    errors="coerce"
)

# Validate conversion
if flow_outliers["Outlier Percentage"].isna().any():
    raise ValueError(
        "One or more outlier-percentage values could not be "
        "converted to numeric format."
    )

# ------------------------------------------------------------
# Preserve intended variable order
# ------------------------------------------------------------

variable_order = [
    "Total Anode Stack Flow",
    "Total Cathode Stack Flow"
]

flow_outliers["Variable"] = pd.Categorical(
    flow_outliers["Variable"],
    categories=variable_order,
    ordered=True
)

flow_outliers = (
    flow_outliers
    .sort_values("Variable")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Create Figure
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    flow_outliers["Variable"],
    flow_outliers["Outlier Percentage"],
    edgecolor="black",
    linewidth=1
)

ax.set_title(
    "IQR Outlier Percentage of Reactant Flow Variables",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Reactant Flow Variable")
ax.set_ylabel("Outliers (%)")

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.5
)

ax.set_axisbelow(True)

# ------------------------------------------------------------
# Dynamic axis limits
# ------------------------------------------------------------

max_value = flow_outliers["Outlier Percentage"].max()

upper_limit = max_value * 1.18 if max_value > 0 else 1

ax.set_ylim(0, upper_limit)

# ------------------------------------------------------------
# Add value labels
# ------------------------------------------------------------

label_offset = upper_limit * 0.02

for bar, value in zip(
    bars,
    flow_outliers["Outlier Percentage"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + label_offset,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

plt.tight_layout()
plt.show()

<Figure size 1080x600 with 1 Axes>

In [167]:
# Notebook 7 completed successfully.